# 03 - Evaluacion del sistema hibrido (reglas + modelo + scoring)

**Proyecto:** AchachAI - hackIAthon 2026 - Reto Aseguradora del Sur

Evaluacion END-TO-END del sistema completo, no solo del modelo XGBoost:
1. Cobertura de las 7 reglas RF-01..RF-07 sobre los casos inyectados
2. Distribucion del score 0-100 y semaforo VERDE/AMARILLO/ROJO
3. Falsos positivos sobre no-fraudes
4. Comparacion: solo reglas vs solo modelo vs sistema hibrido
5. Casos donde el modelo se complementa con las reglas

**Pre-requisitos:** todas las tablas en `data/processed/`, motor de reglas en `src/rules/`, modelo en `runs/local/`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.rules import build_contexto, evaluate_siniestro

PROC = ROOT / 'data' / 'processed'
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

# Cargar 7 tablas
tablas = {n: pd.read_parquet(PROC / f'{n}.parquet') for n in
          ['siniestros','polizas','asegurados','vehiculos','proveedores','conductores','documentos']}
for n, df in tablas.items():
    print(f'{n:<12} {len(df):>7,} filas')

## 2. Evaluar motor de reglas sobre un sample

In [ ]:
# Contexto agregado (precomputado una vez)
ctx = build_contexto(tablas['siniestros'], tablas['proveedores'])
print(f'Contexto listo. Umbral P90 proveedor: {ctx.umbral_recurrencia_proveedor}')
print(f'Proveedores en lista restrictiva: {len(ctx.proveedores_lista_restrictiva)}')

# Indices rapidos
pol_idx = tablas['polizas'].set_index('id_poliza')
ase_idx = tablas['asegurados'].set_index('id_asegurado')
veh_idx = tablas['vehiculos'].set_index('id_vehiculo')
prov_idx = tablas['proveedores'].set_index('id_proveedor')
cond_idx = tablas['conductores'].set_index('id_conductor')
docs_por_sin = tablas['documentos'].groupby('id_siniestro').apply(
    lambda d: d.to_dict('records'), include_groups=False
).to_dict()

# Sample: TODOS los inyectados + 1000 aleatorios
inj = tablas['siniestros'][tablas['siniestros']['caso_inyectado']]
rand = tablas['siniestros'][~tablas['siniestros']['caso_inyectado']].sample(1000, random_state=42)
sample = pd.concat([inj, rand])
print(f'\nSample evaluado: {len(sample)} ({len(inj)} inyectados + 1000 aleatorios)')

In [ ]:
# Evaluar motor de reglas
resultados = []
for _, sin in sample.iterrows():
    try:
        r = evaluate_siniestro(
            siniestro=sin.to_dict(),
            poliza=pol_idx.loc[sin['id_poliza']].to_dict() | {'id_poliza': sin['id_poliza']},
            asegurado=ase_idx.loc[sin['id_asegurado']].to_dict() | {'id_asegurado': sin['id_asegurado']},
            vehiculo=veh_idx.loc[sin['id_vehiculo']].to_dict() | {'id_vehiculo': sin['id_vehiculo']},
            proveedor=prov_idx.loc[sin['id_proveedor']].to_dict() | {'id_proveedor': sin['id_proveedor']},
            conductor=cond_idx.loc[sin['id_conductor']].to_dict() | {'id_conductor': sin['id_conductor']},
            documentos=docs_por_sin.get(sin['id_siniestro'], []),
            ctx=ctx,
        )
        resultados.append({
            'id_siniestro': sin['id_siniestro'],
            'es_inyectado': bool(sin['caso_inyectado']),
            'etiqueta_fraude': int(sin['etiqueta_fraude_simulada']),
            'score': r['score'],
            'nivel': r['nivel'],
            'n_reglas_criticas': len(r['reglas_criticas']),
            'reglas_codigos': ','.join(reg['codigo'] for reg in r['reglas_criticas']),
            'n_senales': len(r['senales_activadas']),
        })
    except Exception:
        continue

res = pd.DataFrame(resultados)
print(f'Evaluados: {len(res)}')
print(f'Distribucion nivel: {res["nivel"].value_counts().to_dict()}')

## 3. Distribucion del score y semaforo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, sub, color in [('Inyectados (esperado ROJO)', res[res['es_inyectado']], '#e74c3c'),
                           ('Aleatorios', res[~res['es_inyectado']], '#3498db')]:
    axes[0].hist(sub['score'], bins=20, alpha=0.6, label=label, color=color)
axes[0].axvline(41, ls='--', color='orange', label='Umbral AMARILLO')
axes[0].axvline(76, ls='--', color='red', label='Umbral ROJO')
axes[0].set_xlabel('Score (0-100)')
axes[0].set_ylabel('# siniestros')
axes[0].set_title('Distribucion del score por grupo')
axes[0].legend()

niveles_inj = res[res['es_inyectado']]['nivel'].value_counts()
niveles_rand = res[~res['es_inyectado']]['nivel'].value_counts()
x = np.arange(3)
axes[1].bar(x - 0.2, [niveles_inj.get(l, 0) for l in ['VERDE','AMARILLO','ROJO']], 0.4, label='Inyectados', color='#e74c3c')
axes[1].bar(x + 0.2, [niveles_rand.get(l, 0) for l in ['VERDE','AMARILLO','ROJO']], 0.4, label='Aleatorios', color='#3498db')
axes[1].set_xticks(x); axes[1].set_xticklabels(['VERDE', 'AMARILLO', 'ROJO'])
axes[1].set_ylabel('# siniestros')
axes[1].set_title('Conteo por nivel')
axes[1].legend()

plt.tight_layout(); plt.show()

## 4. Cobertura por regla critica

In [ ]:
reglas_por_caso = []
for _, r in res.iterrows():
    for code in r['reglas_codigos'].split(','):
        if code:
            reglas_por_caso.append({'regla': code, 'es_inyectado': r['es_inyectado'], 'nivel': r['nivel']})
rdf = pd.DataFrame(reglas_por_caso)

print('Activaciones por regla:')
print(rdf.groupby(['regla','es_inyectado']).size().unstack(fill_value=0).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
rdf.groupby(['regla','es_inyectado']).size().unstack(fill_value=0).plot(kind='bar', stacked=True, ax=ax, color=['#3498db','#e74c3c'])
ax.set_title('Activaciones por regla critica (apilado por origen)')
ax.set_xlabel('Regla')
ax.set_ylabel('# activaciones')
ax.legend(['No inyectado', 'Inyectado'])
plt.tight_layout(); plt.show()

## 5. Tasa de falsos positivos (sobre aleatorios no-fraude)

Idealmente queremos que pocos casos NO-fraude queden marcados como AMARILLO/ROJO.

In [ ]:
no_fraude = res[(~res['es_inyectado']) & (res['etiqueta_fraude'] == 0)]
fp_rojo = (no_fraude['nivel'] == 'ROJO').sum()
fp_amar = (no_fraude['nivel'] == 'AMARILLO').sum()
fp_total = fp_rojo + fp_amar

print(f'No-fraudes evaluados: {len(no_fraude)}')
print(f'  Falsos positivos ROJO:     {fp_rojo} ({fp_rojo/len(no_fraude)*100:.1f}%)')
print(f'  Falsos positivos AMARILLO: {fp_amar} ({fp_amar/len(no_fraude)*100:.1f}%)')
print(f'  Tasa FP total:             {fp_total/len(no_fraude)*100:.1f}%')

# Los falsos positivos rojos vienen mayormente de RF-03 (proveedor en lista restrictiva)
# que es comportamiento correcto: cualquier siniestro de un proveedor restrictivo SI debe revisarse.
fp = no_fraude[no_fraude['nivel'].isin(['ROJO','AMARILLO'])]
print('\nReglas mas comunes en falsos positivos:')
for r in fp['reglas_codigos'].str.split(',').explode().value_counts().head(5).items():
    print(f'  {r[0] or "(sin regla critica)"}: {r[1]}')

## 6. Cobertura: casos inyectados detectados como ROJO

Los 40 casos inyectados deberian quedar como ROJO (RF-01..04) o AMARILLO (RF-05).

In [ ]:
inj_res = res[res['es_inyectado']]
detectados = (inj_res['nivel'] != 'VERDE').sum()
print(f'Inyectados detectados (no-VERDE): {detectados}/{len(inj_res)} = {detectados/len(inj_res)*100:.1f}%')

for nivel in ['ROJO','AMARILLO','VERDE']:
    n = (inj_res['nivel'] == nivel).sum()
    print(f'  {nivel}: {n}/{len(inj_res)} ({n/len(inj_res)*100:.1f}%)')

## 7. Conclusiones del sistema hibrido

**Lo que funciona bien:**
- Las reglas RF-01..05 detectan correctamente los casos inyectados (>97% como AMARILLO o ROJO).
- La explicabilidad es total: cada caso ROJO muestra exactamente que regla(s) y senal(es) disparo.
- El modelo XGBoost complementa con casos que las reglas no atrapan (recall del modelo: 57% en test).

**Limitaciones identificadas:**
- ~10% de falsos positivos en no-fraudes, dominado por RF-03 (proveedor en lista restrictiva). En produccion, RF-03 deberia ser mas granular (asegurado especifico + proveedor) en vez de catch-all.
- Senales 13 (narrativas similares) y RF-07 (clonadas) requieren embeddings - se calculan en `scripts/compute_embeddings.py`.

**Decision arquitectonica:** el sistema final es HIBRIDO:
- **Reglas determinisitcas** = baseline auditable, 100% trazable.
- **Modelo XGBoost** = captura patrones no obvios.
- **Agente GPT-5-mini** = consulta natural + explicacion no-acusatoria.

Ningun analista deberia rechazar un siniestro basado solo en este sistema. El score es una **alerta** que ayuda a priorizar revisiones humanas.